In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)

results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [5]:
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI

from rag_helper import RAGBase

openai_client = OpenAI()
rag = RAGBase(index=index, llm_client=openai_client, model="openai/gpt-5.4-mini")

In [6]:
query = "How does the agentic loop keep calling the model until it stops?"

answer, usage = rag.rag(query)

print(answer)
print("input_tokens:", usage.input_tokens)

The loop keeps calling the model inside a `while True` loop. After each model response, it checks whether there were any `function_call` items:

- if there is a function call, it runs the tool, appends the tool result to `messages`, and continues;
- if there are no function calls, it breaks out of the loop.

So the stopping condition is:

```python
if has_function_calls == False:
    break
```

In other words, it keeps looping until the model returns a final message without asking for any more tools.
input_tokens: 7111


In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [8]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunk_index.fit(chunks)

rag_chunks = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
    model="openai/gpt-5.4-mini",
)

In [9]:
answer_chunked, usage_chunked = rag_chunks.rag(query)

q3_input_tokens = usage.input_tokens
q5_input_tokens = usage_chunked.input_tokens

print(answer_chunked)
print("Q3 input_tokens:", q3_input_tokens)
print("Q5 input_tokens:", q5_input_tokens)
print("Q3 / Q5:", round(q3_input_tokens / q5_input_tokens, 1))

The loop keeps calling the model inside a `while True` loop and checks whether the model made any `function_call`s on that turn.

- If there is a function call, the code runs the tool, appends the result to the message history, and loops again.
- If there are no function calls, `has_function_calls` stays `False`, and the loop `break`s.

So the stop condition is: **no function calls in the model’s response means the agent is done**.
Q3 input_tokens: 7111
Q5 input_tokens: 2294
Q3 / Q5: 3.1


In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

search_call_count = 0


def search(query: str) -> list[dict]:
    """Search the course lesson chunks for entries matching the given query."""
    global search_call_count
    search_call_count += 1
    return chunk_index.search(query, num_results=5)


instructions = (
    "You're a course teaching assistant. Answer the student's question using the "
    "search tool. Make multiple searches with different keywords before answering."
)

agent_tools = Tools()
agent_tools.add_tool(search)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(
        model="openai/gpt-5.4-mini",
        extra_kwargs={"max_output_tokens": 1024},
    ),
)

In [ ]:
search_call_count = 0

agent_question = (
    "How does the agentic loop work, and how is it different from plain RAG?"
)

result = runner.loop(prompt=agent_question)

print(result.last_message)
print("search calls:", search_call_count)